# Daily Challenge — Breast Cancer Prediction

In this challenge, I predict if a breast tumor is malignant (cancer) or benign (not cancer) from measurements of its cells. I use 4 classification models: Logistic Regression, K Nearest Neighbours, Random Forests and Support Vector Machines (SVM), and I find the best one.

### What I learn here
- Data visualization with scatter plots.
- A basic understanding of classification models.
- Interpreting the results.

## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. I download the dataset from [Kaggle (uciml/breast-cancer-wisconsin-data)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data) with the **Download** button. Kaggle gives a zip named `archive.zip`. I rename it to `breast_cancer_data.zip` and keep it in my `C:\Users\sivan\Datasets` folder.
3. I run the cells from top to bottom. When the upload cell opens a file picker, I choose `breast_cancer_data.zip`.

In [ ]:
from google.colab import files
# I import Colab's tool for uploading files from my computer.
uploaded = files.upload()
# I open a file picker and choose "breast_cancer_data.zip".
!unzip -o "breast_cancer_data.zip"
# I extract data.csv from the zip into Colab.

## Exploratory Data Analysis

### Step 1 — Use pandas to load the dataset and examine the first few rows

In this step, I load `data.csv` with pandas and look at its size and first rows.

### What I learn here
- Each row is one tumor: an `id`, the `diagnosis` (M = malignant, B = benign), and 30 measurements of the cell nuclei (mean, standard error and worst value of 10 features).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# I read the CSV file that I extracted from the zip.
df = pd.read_csv('data.csv')
print(df.shape)
display(df.head())

### Step 2 — Check and handle the missing values

In this step, I count the missing values in every column and I handle them.

### What I learn here
- A column that is empty for every row carries no information, so I remove it.

In [ ]:
# I count the missing values per column and show only the columns that have some.
missing = df.isna().sum()
print(missing[missing > 0])

# Kaggle's file ends every line with an extra comma, so pandas creates an empty column 'Unnamed: 32'.
# how='all' drops only the columns where every value is missing.
df = df.dropna(axis=1, how='all')
print('Missing values left:', df.isna().sum().sum())
print(df.shape)

**What I see:** The only column with missing values is `Unnamed: 32`, with 569 missing values out of 569 rows, so it is completely empty. After I drop it, there are **0 missing values** left, and the table has 569 rows and 32 columns.

### Step 3 — Drop any unnecessary column

In this step, I drop the `id` column.

### What I learn here
- `id` is only a number to identify each patient. It says nothing about the tumor, so it would only confuse the models.

In [ ]:
df = df.drop(columns=['id'])
print(df.shape)
print(df.columns.tolist())

### Step 4 — Create a Countplot to display diagnosis from magma

In this step, I draw a countplot of the `diagnosis` column with seaborn's `magma` color palette. I also draw one scatter plot of two features, colored by diagnosis, because this challenge teaches data visualization with scatter plots.

### What I learn here
- A countplot draws one bar per category, with the number of rows in it.
- A scatter plot shows if two features already separate the two classes.

In [ ]:
# countplot counts the rows of each diagnosis. palette='magma' gives the bars the magma colors.
sns.countplot(x='diagnosis', data=df, hue='diagnosis', palette='magma', legend=False)
plt.title('Number of tumors per diagnosis (M = malignant, B = benign)')
plt.show()

# I put the mean radius on the x-axis and the mean texture on the y-axis, with one color per diagnosis.
sns.scatterplot(x='radius_mean', y='texture_mean', hue='diagnosis', data=df, palette='magma', alpha=0.7)
plt.title('Mean radius vs mean texture')
plt.show()

**What I see:** The countplot shows more benign tumors (B) than malignant ones (M). In the scatter plot, the malignant tumors mostly have a bigger mean radius (above about 15) and the benign tumors a smaller one, so even these two features separate the classes quite well, with some overlap in the middle.

## Data Preprocessing, Building Models and Evaluation

### Step 5 — Counts of unique rows in the 'diagnosis' column

In this step, I count how many tumors are malignant and how many are benign.

In [ ]:
# value_counts() counts the rows of every unique value in the column.
print(df['diagnosis'].value_counts())
print(df['diagnosis'].value_counts(normalize=True).round(3))

### Step 6 — Map categorical values to numerical values

In this step, I change the `diagnosis` letters into numbers: M (malignant) becomes 1 and B (benign) becomes 0.

### What I learn here
- Models need numbers, and for a yes/no problem I use 1 for the class I want to find (here, cancer).

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
print(df['diagnosis'].value_counts())

### Step 7 — Splitting the data into train and test

In this step, I separate the features `X` from the target `y`, and I keep 20% of the tumors for testing. I also scale the features, because Logistic Regression, KNN and SVM work much better when all the features are on the same scale.

### What I learn here
- `stratify=y` keeps the same share of malignant tumors in both sets.
- I fit the scaler on the training set only, so the test set stays unseen.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X holds the 30 measurements, y holds the answer (1 = malignant, 0 = benign).
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# For example area_mean goes up to about 2500, but smoothness_mean stays below 0.2, so I scale.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn the mean and std from the training rows, then scale them
X_test_scaled = scaler.transform(X_test)        # scale the test rows with the same numbers
print('Train:', X_train.shape, '| Test:', X_test.shape)

### Step 8 — Implement logistic regression and print the accuracy

In this step, I train a Logistic Regression and print its accuracy on the test set.

### What I learn here
- Logistic Regression draws a straight boundary and gives the probability of cancer for each tumor.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
# Accuracy = correct predictions / all test tumors.
acc_lr = accuracy_score(y_test, log_reg.predict(X_test_scaled))
print('Logistic Regression accuracy:', round(acc_lr, 4))

### Step 9 — Implement K Nearest Neighbours and print the accuracy

In this step, I train a KNN and print its accuracy.

### What I learn here
- KNN looks at the 5 closest tumors in the training set and takes a vote.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)  # 5 neighbors is the default
knn.fit(X_train_scaled, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test_scaled))
print('K Nearest Neighbours accuracy:', round(acc_knn, 4))

### Step 10 — Implement Random Forests and print the accuracy

In this step, I train a Random Forest and print its accuracy.

### What I learn here
- A Random Forest trains many decision trees on random parts of the data, and the trees vote.
- Trees do not need scaled features, but scaling does not change their result, so I use the same scaled data for all models.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)  # 100 trees
rf.fit(X_train_scaled, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test_scaled))
print('Random Forest accuracy:', round(acc_rf, 4))

### Step 11 — Implement Support Vector Machines (SVM) and print the accuracy

In this step, I train an SVM and print its accuracy.

### What I learn here
- The SVM looks for the boundary with the widest margin between the two classes. `kernel='rbf'` allows a curved boundary.

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)  # the default SVM settings
svm.fit(X_train_scaled, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test_scaled))
print('SVM accuracy:', round(acc_svm, 4))

### Step 12 — Which is the best model ?

In this step, I put the four accuracies side by side and I look at the confusion matrix of the best model.

### What I learn here
- For cancer, a missed malignant tumor (false negative) is the worst mistake, so I also check the recall.

In [ ]:
from sklearn.metrics import recall_score, ConfusionMatrixDisplay

models = {'Logistic Regression': log_reg, 'K Nearest Neighbours': knn, 'Random Forest': rf, 'SVM': svm}
results = pd.DataFrame({
    'accuracy': [accuracy_score(y_test, m.predict(X_test_scaled)) for m in models.values()],
    # recall = share of the malignant tumors that the model finds
    'recall (malignant)': [recall_score(y_test, m.predict(X_test_scaled)) for m in models.values()],
}, index=models.keys()).round(4)
display(results.sort_values('accuracy', ascending=False))

sns.barplot(x=results.index, y=results['accuracy'], hue=results.index, palette='magma', legend=False)
plt.ylim(0.9, 1.0)
plt.title('Test accuracy of the 4 models')
plt.xticks(rotation=15)
plt.show()

# idxmax() gives the name of the model with the highest accuracy (the first one if there is a tie).
best_name = results['accuracy'].idxmax()
print('Best model:', best_name)
ConfusionMatrixDisplay.from_estimator(models[best_name], X_test_scaled, y_test, display_labels=['Benign (0)', 'Malignant (1)'])
plt.title(best_name + ' - confusion matrix')
plt.show()

**My answer:** The accuracies are **0.9649** for Logistic Regression, **0.9561** for K Nearest Neighbours, **0.9737** for Random Forest and **0.9737** for SVM. **Random Forest and SVM tie as the best models**, with 111 of the 114 test tumors right. The code prints Random Forest because `idxmax()` takes the first one in a tie. Its confusion matrix shows that all 72 benign tumors are right (0 false alarms) and that it finds 39 of the 42 malignant tumors, so it misses 3 (recall 0.9286). All four models are above 95%, and Logistic Regression has the same recall as the two best models. For cancer, a missed malignant tumor is the worst mistake, so the next thing I would try is to raise the recall.